# 04 — MICM Fuel Prices | Automatic Download (raw landing)

Downloads the MICM weekly fuel-price data from **two origins** and lands it, unchanged, in the raw archive
`/Volumes/rd_economic_intelligence/bronze/micm_raw/fuel_prices/`.

| Origin | What | Why |
|---|---|---|
| Open-data portal (datos.gob.do, CKAN `precios-de-los-combustibles`) | **XLSX** (authoritative) and **CSV** (reconciliation) with the full series since 2010 | Complete history, but the file is updated late: in November 2026 it still ended at the week 31 Oct–6 Nov. |
| MICM weekly notices (`AVISO-PRE.-SEM.CORTE-…pdf`, page *Avisos semanales de precios de combustibles*) | One **PDF** per price week (Saturday to Friday), signed one or two days before the week starts | Official source of the newest weeks. Silver uses it only for weeks the open-data file does not have yet. |

| Step | Rule |
|---|---|
| Discover | Open data: the URL of the XLSX and CSV is read from the CKAN API on every run (the file name carries the year range). Notices: the PDF links are read from the MICM page; notices already archived are not downloaded again. |
| Validate | XLSX must contain `DIA DESDE`, `DIA HASTA`, `MES`, `AÑO` and the main fuels; the CSV header must contain `DIA DESDE`. Each **PDF is parsed before it is archived** (`micm_parsers.parse_notice_pdf`): week dates, the five core fuels and plausible prices (50–1,000 DOP/gal). A notice that fails the check stops the task and is not archived. |
| Land | Shared helper `src/shared/acquisition.py`: identical content is never landed twice; new content goes to `<yyyy>/<mm>/`; a republished file with an archived name is kept as `<name>__<date>`. PDFs go to the same folder as the XLSX and CSV. |
| Source down | Each origin is independent. If one cannot be reached after 3 attempts (network, invalid TLS certificate, HTTP 5xx), the outage is logged in `quality.source_availability` and the other origin still runs. If both are down, the task ends as a warning (`SOURCE_UNAVAILABLE`). HTTP 4xx and layout changes still fail. |
| Signal | Task value `downloaded` = true when at least one new file landed. |

Bronze (`04_MICM_Fuel_Prices_Bronze`) then captures new files incrementally with Auto Loader. **Status:** Development.

In [0]:
%pip install openpyxl==3.1.5 pdfplumber==0.11.10 --quiet

In [0]:
dbutils.library.restartPython()

In [0]:
import sys, io, os, json, importlib, urllib.parse
import pandas as pd

nb = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
sys.path.insert(0, "/Workspace" + nb.rsplit("/", 2)[0] + "/shared")
import acquisition as ACQ
import micm_parsers as MP
importlib.reload(ACQ); importlib.reload(MP)

LANDING = "/Volumes/rd_economic_intelligence/bronze/micm_raw/fuel_prices/"
PACKAGE = "precios-de-los-combustibles"
REQUIRED = ["DIA DESDE", "DIA HASTA", "MES", "AÑO", "GASOLINA PREMIUM", "GASOLINA REGULAR", "GASOIL OPTIMO", "GLP"]
outcome, outages = {}, []

# Origin 1 · open-data portal (XLSX authoritative + CSV for reconciliation)
try:
    urls, portal_modified = ACQ.ckan_resources(PACKAGE, ["xlsx", "csv"])
    print(f"Portal dataset updated {portal_modified}: {urls}")
    files = {fmt: ACQ.http_get(url) for fmt, url in urls.items()}
    cols = [str(c).strip() for c in pd.read_excel(io.BytesIO(files["xlsx"]), dtype=object, nrows=5).columns]
    missing = [c for c in REQUIRED if c not in cols]
    if missing:
        raise ValueError(f"XLSX failed the structural contract, missing columns: {missing}")
    if "DIA DESDE" not in files["csv"][:500].decode("latin-1"):
        raise ValueError("CSV failed the structural contract: header 'DIA DESDE' not found")
    outcome["open_data"] = {"portal_modified": portal_modified,
                            "files": {fmt: ACQ.land(data, ACQ.urllib.parse.unquote(urls[fmt].rsplit("/", 1)[-1]), LANDING) for fmt, data in files.items()}}
except ACQ.SourceUnavailable as e:
    outages.append(("micm_fuel_prices", e))

# Origin 2 · weekly price notices (PDF): newest weeks, ahead of the open-data file
try:
    html = ACQ.http_get(MP.NOTICES_PAGE, timeout=60).decode("utf-8", "ignore")
    links = MP.notice_links(html)
    if not links:
        raise ValueError("no PDF notices found on the MICM page: layout changed")
    archived = {os.path.basename(p) for p in ACQ._archived_files(LANDING)}
    todo = [u for u in links if urllib.parse.unquote(u.rsplit("/", 1)[-1]) not in archived]
    notices = {}
    for url in todo:
        name = urllib.parse.unquote(url.rsplit("/", 1)[-1])
        data = ACQ.http_get(url, timeout=90)
        n = MP.parse_notice_pdf(data)                      # validate before archiving; raises if the layout changed
        notices[name] = {**ACQ.land(data, name, LANDING), "week": f"{n['start']} to {n['end']}", "fuels": len(n["prices"])}
    outcome["notices"] = {"listed": len(links), "already_archived": len(links) - len(todo), "files": notices}
except ACQ.SourceUnavailable as e:
    outages.append(("micm_fuel_notices", e))

for source, e in outages:
    ACQ.log_outage(spark, source, e)
if len(outages) == 2:
    ACQ.source_unavailable(dbutils, spark, "micm_fuel_prices", outages[0][1])   # nothing reachable: end as a warning

landed = [r for o in outcome.values() for r in o["files"].values()]
downloaded = any(r["status"] == "DOWNLOADED" for r in landed)
dbutils.jobs.taskValues.set(key="downloaded", value=downloaded)
dbutils.notebook.exit(json.dumps({"status": "DOWNLOADED" if downloaded else "SKIPPED",
                                  "origins_unavailable": [s for s, _ in outages], **outcome}, ensure_ascii=False, default=str))